In [10]:
from pathlib import Path
import pandas as pd

# Source files for the master food table.
project_root = Path(r"C:\Users\youse\Downloads\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30")
food_path = project_root / "food.csv"
food_nutrient_path = project_root / "food_nutrient.csv"
nutrient_path = project_root / "nutrient.csv"

# Load the USDA tables.
food_all = pd.read_csv(food_path)
food_nutrient = pd.read_csv(food_nutrient_path, low_memory=False)
nutrient = pd.read_csv(nutrient_path)

# Keep only the canonical Foundation Foods records for the master nutrition profile.
food = food_all.loc[food_all["data_type"].eq("foundation_food")].copy()

print("Food rows before filtering:", len(food_all))
print("Food rows after filtering to foundation_food:", len(food))
print("Food nutrient shape:", food_nutrient.shape)
print("Nutrient shape:", nutrient.shape)
print()
print("Food data_type counts after filtering:")
print(food["data_type"].value_counts().to_string())
print()
print("Food columns:", list(food.columns))
print("Food nutrient columns:", list(food_nutrient.columns))
print("Nutrient columns:", list(nutrient.columns))

# Resolve the nutrient rows needed for the final master table from the lookup table itself.
# Using exact normalized name matches keeps the lookup deterministic and avoids regex warnings.
target_nutrients = {
    "Calories": ["energy", "energy (atwater general factors)", "energy (atwater specific factors)"],
    "Protein": ["protein"],
    "Carbs": ["carbohydrate, by difference", "carbohydrate, by summation", "carbohydrates"],
    "Fat": ["total lipid (fat)", "total fat (nlea)"],
    "Fiber": ["fiber, total dietary", "total dietary fiber (aoac 2011.25)", "fiber, insoluble", "fiber, soluble"],
    "Sugar": ["total sugars", "sugars, total", "sucrose"],
    "Sodium": ["sodium, na"],
}

nutrient_lower = nutrient["name"].astype(str).str.lower().str.strip()
for label, candidates in target_nutrients.items():
    candidate_set = {candidate.lower() for candidate in candidates}
    matches = nutrient.loc[nutrient_lower.isin(candidate_set), ["id", "name", "unit_name", "nutrient_nbr", "rank"]].copy()
    print(f"\n{label} matches:")
    if matches.empty:
        print("  No matches found")
    else:
        print(matches.to_string(index=False))


Food rows before filtering: 87990
Food rows after filtering to foundation_food: 469
Food nutrient shape: (170469, 11)
Nutrient shape: (477, 5)

Food data_type counts after filtering:
data_type
foundation_food    469

Food columns: ['fdc_id', 'data_type', 'description', 'food_category_id', 'publication_date']
Food nutrient columns: ['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id', 'min', 'max', 'median', 'footnote', 'min_year_acquired']
Nutrient columns: ['id', 'name', 'unit_name', 'nutrient_nbr', 'rank']

Calories matches:
  id                              name unit_name  nutrient_nbr  rank
2047  Energy (Atwater General Factors)      KCAL         957.0 280.0
2048 Energy (Atwater Specific Factors)      KCAL         958.0 290.0
1008                            Energy      KCAL         208.0 300.0
1062                            Energy        kJ         268.0 400.0

Protein matches:
  id    name unit_name  nutrient_nbr  rank
1003 Protein         G         203.0 600.

In [11]:
food["data_type"].value_counts()

data_type
foundation_food    469
Name: count, dtype: int64

In [12]:
# Merge food_nutrient with nutrient only so we can work with nutrient names instead of IDs.
def merge_food_nutrient_with_nutrients(food_nutrient_df: pd.DataFrame, nutrient_df: pd.DataFrame) -> pd.DataFrame:
    """Return food nutrient rows enriched with nutrient metadata."""
    nutrient_lookup = nutrient_df[["id", "name", "unit_name"]].rename(columns={"id": "nutrient_lookup_id"}).copy()
    merged = food_nutrient_df.merge(
        nutrient_lookup,
        left_on="nutrient_id",
        right_on="nutrient_lookup_id",
        how="left",
        validate="m:1",
    )
    merged = merged.rename(columns={"name": "nutrient_name", "unit_name": "nutrient_unit"})
    merged = merged.drop(columns=["nutrient_lookup_id"])
    return merged


def validate_merge_row_count(original_df: pd.DataFrame, merged_df: pd.DataFrame) -> None:
    """Fail fast if the merge changes the number of rows."""
    original_rows = len(original_df)
    merged_rows = len(merged_df)
    print(f"Original food_nutrient rows: {original_rows}")
    print(f"Merged rows: {merged_rows}")
    assert original_rows == merged_rows, "Row count changed after merge"


food_nutrient_enriched = merge_food_nutrient_with_nutrients(food_nutrient, nutrient)
validate_merge_row_count(food_nutrient, food_nutrient_enriched)

missing_nutrient_names = food_nutrient_enriched["nutrient_name"].isna().sum()
print(f"Rows with missing nutrient names: {missing_nutrient_names}")

if missing_nutrient_names:
    missing_ids = (
        food_nutrient_enriched.loc[food_nutrient_enriched["nutrient_name"].isna(), "nutrient_id"]
        .dropna()
        .astype("Int64")
        .astype(str)
        .sort_values()
        .unique()
    )
    print("Unmatched nutrient IDs:", ", ".join(missing_ids))

sample_columns = ["fdc_id", "nutrient_id", "nutrient_name", "amount", "nutrient_unit"]
print("\nSample merged rows:")
display(food_nutrient_enriched[sample_columns].head(10))

Original food_nutrient rows: 170469
Merged rows: 170469
Rows with missing nutrient names: 33
Unmatched nutrient IDs: 2066

Sample merged rows:


,fdc_id,nutrient_id,nutrient_name,amount,nutrient_unit
0,319877,1051,Water,56.30,G
1,319877,1002,Nitrogen,1.28,G
2,319877,1004,Total lipid (fat),19.00,G
3,319877,1007,Ash,1.98,G
4,319878,1091,"Phosphorus, P",188.00,MG
5,319878,1101,"Manganese, Mn",1.21,MG
6,319878,1092,"Potassium, K",326.00,MG
7,319878,1087,"Calcium, Ca",46.00,MG
8,319878,1093,"Sodium, Na",446.00,MG
9,319878,1090,"Magnesium, Mg",75.90,MG


In [ ]:
# Merge food with the enriched nutrient rows so each record also carries the food name.
def merge_food_with_enriched_nutrients(food_df: pd.DataFrame, enriched_nutrients_df: pd.DataFrame) -> pd.DataFrame:
    """Return food-nutrient rows with food metadata attached."""
    food_lookup = food_df[["fdc_id", "description"]].rename(columns={"description": "food_name"}).copy()
    merged = enriched_nutrients_df.merge(
        food_lookup,
        on="fdc_id",
        how="left",
        validate="m:1",
    )
    return merged


def validate_food_merge_row_count(original_df: pd.DataFrame, merged_df: pd.DataFrame) -> None:
    """Fail fast if the food merge changes the number of rows."""
    original_rows = len(original_df)
    merged_rows = len(merged_df)
    print(f"Original enriched nutrient rows: {original_rows}")
    print(f"Merged rows with food: {merged_rows}")
    assert original_rows == merged_rows, "Row count changed after merging food"


food_nutrient_with_food = merge_food_with_enriched_nutrients(food, food_nutrient_enriched)
validate_food_merge_row_count(food_nutrient_enriched, food_nutrient_with_food)

missing_food_names = food_nutrient_with_food["food_name"].isna().sum()
print(f"Rows with missing food names: {missing_food_names}")

sample_columns = ["fdc_id", "food_name", "nutrient_name", "amount", "nutrient_unit"]
print("\nSample rows after merging food:")
display(food_nutrient_with_food[sample_columns].head(10))

Original enriched nutrient rows: 170469
Merged rows with food: 170469
Rows with missing food names: 149010

Sample rows after merging food:


,fdc_id,food_name,nutrient_name,amount,nutrient_unit
0,319877,NaN,Water,56.30,G
1,319877,NaN,Nitrogen,1.28,G
2,319877,NaN,Total lipid (fat),19.00,G
3,319877,NaN,Ash,1.98,G
4,319878,NaN,"Phosphorus, P",188.00,MG
5,319878,NaN,"Manganese, Mn",1.21,MG
6,319878,NaN,"Potassium, K",326.00,MG
7,319878,NaN,"Calcium, Ca",46.00,MG
8,319878,NaN,"Sodium, Na",446.00,MG
9,319878,NaN,"Magnesium, Mg",75.90,MG


In [5]:
# Filter the merged dataset down to the nutrients required by the application.
def filter_required_nutrients(enriched_food_df: pd.DataFrame) -> pd.DataFrame:
    """Keep only the nutrient rows needed for the downstream master table."""
    required_nutrient_map = {
        "Energy (Atwater General Factors)": ("Calories", 1),
        "Energy (Atwater Specific Factors)": ("Calories", 2),
        "Energy": ("Calories", 3),
        "Protein": ("Protein", 1),
        "Total lipid (fat)": ("Fat", 1),
        "Total fat (NLEA)": ("Fat", 2),
        "Carbohydrate, by difference": ("Carbs", 1),
        "Carbohydrate, by summation": ("Carbs", 2),
        "Carbohydrates": ("Carbs", 3),
        "Fiber, total dietary": ("Fiber", 1),
        "Total dietary fiber (AOAC 2011.25)": ("Fiber", 2),
        "Total Sugars": ("Sugar", 1),
        "Sugars, Total": ("Sugar", 2),
        "Sodium, Na": ("Sodium", 1),
    }

    selected = enriched_food_df.loc[enriched_food_df["nutrient_name"].isin(required_nutrient_map)].copy()
    selected["source_nutrient_name"] = selected["nutrient_name"]
    selected["nutrient_name"] = selected["source_nutrient_name"].map(lambda nutrient: required_nutrient_map[nutrient][0])
    selected["nutrient_priority"] = selected["source_nutrient_name"].map(lambda nutrient: required_nutrient_map[nutrient][1])
    selected = selected.sort_values(["fdc_id", "nutrient_name", "nutrient_priority"], kind="stable")
    return selected


def validate_required_nutrients_only(filtered_df: pd.DataFrame) -> None:
    """Assert that only the requested nutrient names remain after filtering."""
    allowed_nutrient_names = {
        "Calories",
        "Protein",
        "Carbs",
        "Fat",
        "Fiber",
        "Sugar",
        "Sodium",
    }
    remaining_nutrients = set(filtered_df["nutrient_name"].dropna().unique())
    unexpected_nutrients = remaining_nutrients - allowed_nutrient_names
    print("Remaining nutrient names:", sorted(remaining_nutrients))
    print("Selected source nutrient names and counts:")
    print(filtered_df["source_nutrient_name"].value_counts().to_string())
    assert not unexpected_nutrients, f"Unexpected nutrient names found: {sorted(unexpected_nutrients)}"


rows_before_filter = len(food_nutrient_with_food)
filtered_food_nutrient = filter_required_nutrients(food_nutrient_with_food)
rows_after_filter = len(filtered_food_nutrient)

print(f"Rows before filtering: {rows_before_filter}")
print(f"Rows after filtering: {rows_after_filter}")

validate_required_nutrients_only(filtered_food_nutrient)

sample_columns = ["fdc_id", "food_name", "nutrient_name", "source_nutrient_name", "amount", "nutrient_unit"]
print("\nSample filtered rows:")
display(filtered_food_nutrient[sample_columns].head(10))

Rows before filtering: 170469
Rows after filtering: 14580
Remaining nutrient names: ['Calories', 'Carbs', 'Fat', 'Fiber', 'Protein', 'Sodium', 'Sugar']
Selected source nutrient names and counts:
source_nutrient_name
Total lipid (fat)                     4756
Sodium, Na                            3856
Fiber, total dietary                  2472
Protein                               1410
Carbohydrate, by difference            377
Energy (Atwater General Factors)       347
Energy (Atwater Specific Factors)      312
Total dietary fiber (AOAC 2011.25)     309
Sugars, Total                          280
Energy                                 270
Total fat (NLEA)                        99
Carbohydrate, by summation              47
Total Sugars                            45

Sample filtered rows:


,fdc_id,food_name,nutrient_name,source_nutrient_name,amount,nutrient_unit
2,319877,NaN,Fat,Total lipid (fat),19.0,G
8,319878,NaN,Sodium,"Sodium, Na",446.0,MG
16,319882,NaN,Fat,Total lipid (fat),18.7,G
19,319884,NaN,Sodium,"Sodium, Na",444.0,MG
28,319892,NaN,Fat,Total lipid (fat),16.6,G
31,319893,NaN,Sodium,"Sodium, Na",387.0,MG
43,319899,NaN,Fat,Total lipid (fat),19.1,G
50,319900,NaN,Sodium,"Sodium, Na",489.0,MG
97,319908,NaN,Fat,Total lipid (fat),18.2,G
107,319912,NaN,Sodium,"Sodium, Na",428.0,MG


In [6]:
# Build the Master Food Table with one row per food using pivot_table().
filtered_df = globals().get("filtered_food_nutrients", globals().get("filtered_food_nutrient"))

master_food_table = (
    filtered_df.sort_values(["fdc_id", "nutrient_name", "nutrient_priority"], kind="stable")
    .pivot_table(
        index=["fdc_id", "food_name"],
        columns="nutrient_name",
        values="amount",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={"fdc_id": "Food ID", "food_name": "Food Name"})
)

final_columns = ["Food ID", "Food Name", "Calories", "Protein", "Carbs", "Fat", "Fiber", "Sugar", "Sodium"]
master_food_table = master_food_table[final_columns]

print("Master Food Table shape:", master_food_table.shape)
print("Master Food Table columns:", list(master_food_table.columns))
print("\nSample Master Food Table rows:")
display(master_food_table.head(10))

Master Food Table shape: (460, 9)
Master Food Table columns: ['Food ID', 'Food Name', 'Calories', 'Protein', 'Carbs', 'Fat', 'Fiber', 'Sugar', 'Sodium']

Sample Master Food Table rows:


nutrient_name,Food ID,Food Name,Calories,Protein,Carbs,Fat,Fiber,Sugar,Sodium
0,321358,"Hummus, commercial",243.0,7.35,14.90,17.10,5.4,0.34,438.0
1,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",50.0,3.35,4.91,1.90,NaN,4.89,39.0
2,321360,"Tomatoes, grape, raw",31.0,0.83,5.51,0.63,2.1,NaN,6.0
3,321505,"Salt, table, iodized",NaN,NaN,NaN,NaN,NaN,NaN,38700.0
4,321611,"Beans, snap, green, canned, regular pack, drai...",24.0,1.04,4.11,0.39,NaN,1.29,282.0
5,321900,"Broccoli, raw",132.0,2.57,6.29,0.34,2.4,1.40,36.0
6,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",179.0,3.38,5.19,0.95,NaN,4.96,39.0
7,322559,"Milk, nonfat, fluid, with added vitamin A and ...",143.0,3.43,4.89,0.08,NaN,5.05,41.0
8,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",60.0,3.28,4.67,3.20,NaN,4.81,38.0
9,323121,"Frankfurter, beef, unheated",310.0,11.70,2.89,28.00,NaN,1.26,872.0


In [15]:
# Final validation, cleaning, preview, and export for the Master Food Table.
from pathlib import Path

required_columns = [
    "Food ID",
    "Food Name",
    "Calories",
    "Protein",
    "Carbs",
    "Fat",
    "Fiber",
    "Sugar",
    "Sodium",
]

print("Master Food Table shape:", master_food_table.shape)

missing_required_columns = [column for column in required_columns if column not in master_food_table.columns]
print("Missing required columns:", missing_required_columns)
assert not missing_required_columns, f"Missing required columns: {missing_required_columns}"

duplicate_food_ids = master_food_table["Food ID"].duplicated().sum()
duplicate_food_names = master_food_table["Food Name"].duplicated().sum()
print("Duplicate Food IDs:", duplicate_food_ids)
print("Duplicate Food Names:", duplicate_food_names)

missing_values_before = master_food_table.isna().sum().sort_values(ascending=False)
print("\nMissing values before cleaning:")
print(missing_values_before.to_string())

numeric_columns = master_food_table.select_dtypes(include="number").columns.tolist()
print("\nSummary statistics for numeric columns:")
display(master_food_table[numeric_columns].describe())

nutrient_columns = [column for column in required_columns if column not in ["Food ID", "Food Name"]]
missing_nutrient_counts = master_food_table[nutrient_columns].isna().sum()
if missing_nutrient_counts.any():
    print("\nMissing nutrient values are present because the USDA source does not provide every nutrient for every food.")
    print("Missing nutrient counts:")
    print(missing_nutrient_counts[missing_nutrient_counts > 0].to_string())
else:
    print("\nNo missing nutrient values found in the required nutrient columns.")

master_food_table["Food ID"] = master_food_table["Food ID"].astype("Int64")
for column in nutrient_columns:
    master_food_table[column] = pd.to_numeric(master_food_table[column], errors="coerce")

print("\nFirst 10 rows:")
display(master_food_table.head(10))

print("\nLast 10 rows:")
display(master_food_table.tail(10))

sample_size = min(5, len(master_food_table))
print(f"\nFive randomly selected foods (or fewer if the table is smaller):")
display(master_food_table.sample(n=sample_size, random_state=42))

output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "master_food_table.csv"

try:
    master_food_table.to_csv(output_path, index=False)
    saved_path = output_path
except PermissionError:
    fallback_path = output_dir / "master_food_table_pending.csv"
    master_food_table.to_csv(fallback_path, index=False)
    saved_path = fallback_path
    print(f"\nThe requested file is locked by another process: {output_path}")
    print(f"Saved the corrected dataset to: {fallback_path}")
    print("Close the locked CSV and rerun this cell to write the final filename.")

file_size_bytes = saved_path.stat().st_size
print("\nFile location:", saved_path)
print("Number of foods:", len(master_food_table))
print("Number of columns:", master_food_table.shape[1])
print("File size (bytes):", file_size_bytes)
print("Success: master_food_table.csv export step completed.")

Master Food Table shape: (460, 9)
Missing required columns: []
Duplicate Food IDs: 0
Duplicate Food Names: 67

Missing values before cleaning:
nutrient_name
Sugar        275
Fiber        202
Carbs         83
Calories      82
Sodium        57
Fat           39
Protein       35
Food ID        0
Food Name      0

Summary statistics for numeric columns:


nutrient_name,Food ID,Calories,Protein,Carbs,Fat,Fiber,Sugar,Sodium
count,460.0,378.000000,425.000000,377.000000,421.000000,258.000000,185.000000,403.000000
mean,1741121.891304,199.402445,10.569646,17.709922,9.550626,4.248003,8.777056,358.023253
std,977129.66125,215.300499,10.502692,24.417407,19.076417,4.313797,14.541278,2742.748688
min,321358.0,0.000000,0.000000,-0.705000,0.000000,0.000000,0.000000,0.000000
25%,746780.75,46.098775,1.060000,2.440000,0.306300,1.800000,2.218000,0.870650
50%,2258589.5,116.298800,7.350000,7.137000,1.280000,3.129000,4.862000,23.000000
75%,2685569.25,351.050000,19.000000,19.292275,7.308000,4.653000,9.690000,111.400000
max,2768188.0,1640.000000,79.900000,99.600000,99.100000,34.950000,99.800000,38700.000000



Missing nutrient values are present because the USDA source does not provide every nutrient for every food.
Missing nutrient counts:
nutrient_name
Calories     82
Protein      35
Carbs        83
Fat          39
Fiber       202
Sugar       275
Sodium       57

First 10 rows:


nutrient_name,Food ID,Food Name,Calories,Protein,Carbs,Fat,Fiber,Sugar,Sodium
0,321358,"Hummus, commercial",243.0,7.35,14.90,17.10,5.4,0.34,438.0
1,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",50.0,3.35,4.91,1.90,NaN,4.89,39.0
2,321360,"Tomatoes, grape, raw",31.0,0.83,5.51,0.63,2.1,NaN,6.0
3,321505,"Salt, table, iodized",NaN,NaN,NaN,NaN,NaN,NaN,38700.0
4,321611,"Beans, snap, green, canned, regular pack, drai...",24.0,1.04,4.11,0.39,NaN,1.29,282.0
5,321900,"Broccoli, raw",132.0,2.57,6.29,0.34,2.4,1.40,36.0
6,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",179.0,3.38,5.19,0.95,NaN,4.96,39.0
7,322559,"Milk, nonfat, fluid, with added vitamin A and ...",143.0,3.43,4.89,0.08,NaN,5.05,41.0
8,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",60.0,3.28,4.67,3.20,NaN,4.81,38.0
9,323121,"Frankfurter, beef, unheated",310.0,11.70,2.89,28.00,NaN,1.26,872.0



Last 10 rows:


nutrient_name,Food ID,Food Name,Calories,Protein,Carbs,Fat,Fiber,Sugar,Sodium
450,2758993,"Bread, white, commercial",NaN,NaN,NaN,NaN,NaN,NaN,419.700
451,2758994,"Bread, 100% whole wheat, commercial",NaN,NaN,NaN,NaN,NaN,NaN,408.200
452,2758995,"Bread, mulitgrain, commercial",NaN,NaN,NaN,NaN,NaN,NaN,400.500
453,2758996,"Tortilla, wheat flour, shelf stable",NaN,NaN,NaN,NaN,NaN,NaN,730.500
454,2758997,"Tortilla, corn, shelf stable",NaN,NaN,NaN,NaN,NaN,NaN,34.910
455,2758998,"Pasta, dry, enriched, spaghetti",NaN,NaN,NaN,NaN,NaN,NaN,2.406
456,2759000,"Pasta, dry, whole grain, spaghetti",NaN,NaN,NaN,NaN,NaN,NaN,4.583
457,2759001,"Lunchmeat, turkey, oven roasted, sliced",NaN,NaN,NaN,NaN,NaN,0.9866,NaN
458,2759004,"Lunchmeat, chicken breast, sliced",NaN,NaN,NaN,NaN,NaN,1.1400,NaN
459,2768188,"Alaska Pollock, raw",78.4785,17.33,0.0565,0.9925,0.0,NaN,114.800



Five randomly selected foods (or fewer if the table is smaller):


nutrient_name,Food ID,Food Name,Calories,Protein,Carbs,Fat,Fiber,Sugar,Sodium
124,747432,"Beans, Dry, Flor de Mayo (0% moisture)",NaN,23.300000,NaN,0.860,4.000,NaN,NaN
30,326196,"Kale, frozen, cooked, boiled, drained, without...",44.000,2.940000,5.300000,1.210,NaN,1.1200,16.00
199,1999630,"Soy milk, unsweetened, plain, shelf stable",38.485,3.546875,1.293125,2.125,0.000,0.5569,34.28
439,2758978,"Plums, dried (prunes), uncooked",NaN,NaN,NaN,NaN,6.251,31.3800,0.85
154,790085,"Flour, whole wheat, unenriched",370.000,15.100000,71.200000,2.730,10.600,NaN,3.00



File location: C:\Users\youse\Downloads\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\data\processed\master_food_table.csv
Number of foods: 460
Number of columns: 9
File size (bytes): 36883
Success: master_food_table.csv export step completed.
